# TAMP-OS — Batch GUI (updated)
Select multiple microscopy images and convert them all to STL/3MF/GLB in one go.

> ⚠️ Run in **Jupyter Lab or Jupyter Notebook** — not VS Code notebook viewer (tkinter needs a real display).

### Install dependencies (once)

In [ ]:
!pip install numpy pillow scipy numpy-stl

### Imports & pipeline functions

In [9]:
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
from pathlib import Path
import threading, sys
import numpy as np
from PIL import Image
from scipy.ndimage import gaussian_filter
from stl import mesh

In [10]:
import os, sys, struct, json, zipfile
from pathlib import Path
import numpy as np
from PIL import Image
from scipy.ndimage import gaussian_filter
from stl import mesh

BEST_PRACTICE = {"base_thickness": 1.0, "relief_height": 3.0, "blur": 1.2}

def smart_resolution(img_path):
    try:
        img = Image.open(img_path)
        w, h = img.size
        mp = w * h
        if mp >= 4_000_000:   return 512
        elif mp >= 1_000_000: return 384
        else:                 return 256
    except Exception:
        return 256

def smart_size_y(img_path, size_x=100.0):
    try:
        img = Image.open(img_path)
        w, h = img.size
        return round(size_x * h / w, 1)
    except Exception:
        return 75.0

def image_to_heightmap(image_path, output_size=(512,512), invert=False,
                       blur_sigma=1.0, contrast_percentile=(2.0,98.0),
                       preserve_aspect=True, flip=True):
    img = Image.open(image_path).convert("L")
    orig_w, orig_h = img.size
    if preserve_aspect and orig_w != orig_h:
        target_w = output_size[0]
        target_h = round(target_w * orig_h / orig_w)
        actual_size = (target_w, target_h)
        print(f"  [i] {orig_w}x{orig_h} -> {target_w}x{target_h}")
    else:
        actual_size = output_size
    img = img.resize(actual_size, Image.LANCZOS)
    arr = np.array(img, dtype=np.float32)
    lo = np.percentile(arr, contrast_percentile[0])
    hi = np.percentile(arr, contrast_percentile[1])
    arr = np.clip((arr - lo) / (hi - lo + 1e-8), 0.0, 1.0)
    if blur_sigma > 0:
        arr = gaussian_filter(arr, sigma=blur_sigma)
    if invert:
        arr = 1.0 - arr
    if flip:
        arr = np.flipud(arr)
    return arr

def heightmap_to_stl(heightmap, output_path, base_thickness_mm=1.0,
                     max_relief_mm=3.0, physical_size_mm=(100.0,100.0)):
    rows, cols = heightmap.shape
    dx = physical_size_mm[0] / (cols - 1)
    dy = physical_size_mm[1] / (rows - 1)
    hm_ratio = cols / rows
    print_ratio = physical_size_mm[0] / physical_size_mm[1]
    if abs(hm_ratio - print_ratio) > 0.05:
        print(f"  [WARNING] Aspect mismatch — consider size_y={round(physical_size_mm[0]/hm_ratio,1)}")
    z_top = base_thickness_mm + heightmap * max_relief_mm
    num_tris = (rows-1)*(cols-1)*4 + 2*((rows-1)+(cols-1))*2
    litho_mesh = mesh.Mesh(np.zeros(num_tris, dtype=mesh.Mesh.dtype))
    tri_idx = 0
    def add_tri(v0, v1, v2):
        nonlocal tri_idx
        litho_mesh.vectors[tri_idx] = [v0, v1, v2]
        tri_idx += 1
    for r in range(rows-1):
        for c in range(cols-1):
            x0,y0 = c*dx, r*dy
            x1 = (c+1)*dx
            x2,y2 = c*dx, (r+1)*dy
            x3,y3 = (c+1)*dx, (r+1)*dy
            z00,z10,z01,z11 = z_top[r,c],z_top[r,c+1],z_top[r+1,c],z_top[r+1,c+1]
            add_tri([x0,y0,z00],[x1,y0,z10],[x2,y2,z01])
            add_tri([x1,y0,z10],[x3,y3,z11],[x2,y2,z01])
            add_tri([x0,y0,0],[x2,y2,0],[x1,y0,0])
            add_tri([x1,y0,0],[x2,y2,0],[x3,y3,0])
    xmax,ymax = (cols-1)*dx,(rows-1)*dy
    for c in range(cols-1):
        x0,x1 = c*dx,(c+1)*dx
        z0,z1 = z_top[0,c],z_top[0,c+1]
        add_tri([x0,0,0],[x1,0,0],[x0,0,z0])
        add_tri([x1,0,0],[x1,0,z1],[x0,0,z0])
    for c in range(cols-1):
        x0,x1 = c*dx,(c+1)*dx
        z0,z1 = z_top[-1,c],z_top[-1,c+1]
        add_tri([x0,ymax,0],[x0,ymax,z0],[x1,ymax,0])
        add_tri([x1,ymax,0],[x0,ymax,z0],[x1,ymax,z1])
    for r in range(rows-1):
        y0,y1 = r*dy,(r+1)*dy
        z0,z1 = z_top[r,0],z_top[r+1,0]
        add_tri([0,y0,0],[0,y0,z0],[0,y1,0])
        add_tri([0,y1,0],[0,y0,z0],[0,y1,z1])
    for r in range(rows-1):
        y0,y1 = r*dy,(r+1)*dy
        z0,z1 = z_top[r,-1],z_top[r+1,-1]
        add_tri([xmax,y0,0],[xmax,y1,0],[xmax,y0,z0])
        add_tri([xmax,y1,0],[xmax,y1,z1],[xmax,y0,z0])
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    litho_mesh.save(str(output_path))
    print(f"  [OK] STL -> {output_path}")
    return Path(output_path)

def stl_to_3mf(stl_path):
    from stl import mesh as stl_mesh
    m = stl_mesh.Mesh.from_file(str(stl_path))
    vertices = []; vert_map = {}; indices = []
    for tri in m.vectors:
        face = []
        for v in tri:
            key = tuple(float(x) for x in v)
            if key not in vert_map:
                vert_map[key] = len(vertices)
                vertices.append(key)
            face.append(vert_map[key])
        indices.append(face)
    vx = " ".join(
        f'<vertex x="{v[0]:.4f}" y="{v[1]:.4f}" z="{v[2]:.4f}"/>' for v in vertices)
    tx = " ".join(
        f'<triangle v1="{t[0]}" v2="{t[1]}" v3="{t[2]}"/>' for t in indices)
    ns = "http://schemas.microsoft.com/3dmanufacturing/core/2015/02"
    xml = (f'<?xml version="1.0" encoding="UTF-8"?>'
           f'<model unit="millimeter" xmlns="{ns}">'
           f'<resources><object id="1" type="model"><mesh>'
           f'<vertices>{vx}</vertices><triangles>{tx}</triangles>'
           f'</mesh></object></resources><build><item objectid="1"/></build></model>')
    out = Path(str(stl_path).replace(".stl", ".3mf"))
    ct_ns = "http://schemas.openxmlformats.org/package/2006/content-types"
    rel_ns = "http://schemas.openxmlformats.org/package/2006/relationships"
    rel_type = "http://schemas.microsoft.com/3dmanufacturing/2013/01/3dmodel"
    with zipfile.ZipFile(str(out), "w", zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("3D/3dmodel.model", xml)
        zf.writestr("[Content_Types].xml",
            f'<?xml version="1.0"?><Types xmlns="{ct_ns}">'
            f'<Default Extension="rels" ContentType="application/vnd.openxmlformats-package.relationships+xml"/>'
            f'<Default Extension="model" ContentType="application/vnd.ms-package.3dmanufacturing-3dmodel+xml"/>'
            f'</Types>')
        zf.writestr("_rels/.rels",
            f'<?xml version="1.0"?><Relationships xmlns="{rel_ns}">'
            f'<Relationship Type="{rel_type}" Target="/3D/3dmodel.model" Id="r1"/>'
            f'</Relationships>')
    print(f"  [OK] 3MF -> {out}")
    return out

def stl_to_glb(stl_path):
    from stl import mesh as stl_mesh
    m = stl_mesh.Mesh.from_file(str(stl_path))
    verts = m.vectors.reshape(-1, 3).astype(np.float32)
    indices = np.arange(len(verts), dtype=np.uint32)
    def pad4(b): return b + b'\x00' * ((4 - len(b) % 4) % 4)
    ib = pad4(indices.tobytes())
    vb = pad4(verts.tobytes())
    bd = ib + vb
    gltf = {
        "asset": {"version": "2.0", "generator": "TAMP-OS"}, "scene": 0,
        "scenes": [{"nodes": [0]}], "nodes": [{"mesh": 0}],
        "meshes": [{"primitives": [{"attributes": {"POSITION": 1}, "indices": 0}]}],
        "accessors": [
            {"bufferView": 0, "componentType": 5125, "count": len(indices), "type": "SCALAR"},
            {"bufferView": 1, "componentType": 5126, "count": len(verts), "type": "VEC3",
             "min": verts.min(0).tolist(), "max": verts.max(0).tolist()}],
        "bufferViews": [
            {"buffer": 0, "byteOffset": 0, "byteLength": len(ib), "target": 34963},
            {"buffer": 0, "byteOffset": len(ib), "byteLength": len(vb), "target": 34962}],
        "buffers": [{"byteLength": len(bd)}]
    }
    jb = pad4(json.dumps(gltf, separators=(",", ":")).encode())
    def chunk(d, t): return struct.pack("<II", len(d), t) + d
    out = Path(str(stl_path).replace(".stl", ".glb"))
    with open(str(out), "wb") as f:
        f.write(struct.pack("<III", 0x46546C67, 2, 12 + 8 + len(jb) + 8 + len(bd))
                + chunk(jb, 0x4E4F534A) + chunk(bd, 0x004E4942))
    print(f"  [OK] GLB -> {out}")
    return out

def convert_stl(stl_path, formats):
    if formats.get("3mf"): stl_to_3mf(stl_path)
    if formats.get("glb"):  stl_to_glb(stl_path)
    if not formats.get("stl"): stl_path.unlink(missing_ok=True)

print("All pipeline functions loaded OK.")


All pipeline functions loaded OK.


## ▶ Launch the Batch GUI
Run this cell to open the window.

In [11]:
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
import threading

class TAMPBatchGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("TAMP-OS Batch Lithograph Maker")
        self.root.resizable(False, False)
        self.image_paths = []
        self.running = False
        self.advanced_open = tk.BooleanVar(value=False)
        self._build_ui()

    def _build_ui(self):
        pad = dict(padx=10, pady=5)

        tk.Label(self.root, text="TAMP-OS Batch Lithograph Maker",
                 font=("Helvetica", 14, "bold")).grid(row=0, column=0, columnspan=3, pady=(14,2))
        tk.Label(self.root, text="Select microscopy images and convert them all to STL in one go.",
                 font=("Helvetica", 9), fg="gray").grid(row=1, column=0, columnspan=3, pady=(0,10))

        tk.Label(self.root, text="Images:", font=("Helvetica", 10, "bold")).grid(
            row=2, column=0, sticky="nw", **pad)
        fl = tk.Frame(self.root)
        fl.grid(row=2, column=1, columnspan=2, **pad)
        sb = tk.Scrollbar(fl, orient=tk.VERTICAL)
        self.listbox = tk.Listbox(fl, width=50, height=6,
                                  yscrollcommand=sb.set, selectmode=tk.EXTENDED)
        sb.config(command=self.listbox.yview)
        self.listbox.pack(side=tk.LEFT, fill=tk.BOTH)
        sb.pack(side=tk.RIGHT, fill=tk.Y)

        bf = tk.Frame(self.root)
        bf.grid(row=3, column=1, columnspan=2, sticky="w", padx=10)
        tk.Button(bf, text="+ Add Images", command=self._add_images, width=14).pack(side=tk.LEFT, padx=(0,5))
        tk.Button(bf, text="- Remove Selected", command=self._remove_selected, width=14).pack(side=tk.LEFT)

        tk.Label(self.root, text="Output folder:", font=("Helvetica", 10, "bold")).grid(
            row=4, column=0, sticky="w", **pad)
        self.output_var = tk.StringVar(value=str(Path.home() / "TAMP_output"))
        tk.Entry(self.root, textvariable=self.output_var, width=38).grid(
            row=4, column=1, sticky="w", **pad)
        tk.Button(self.root, text="Browse", command=self._browse_output).grid(
            row=4, column=2, sticky="w", padx=(0,10))

        tk.Label(self.root, text="Output format:", font=("Helvetica", 10, "bold")).grid(
            row=5, column=0, sticky="w", **pad)
        ff = tk.Frame(self.root)
        ff.grid(row=5, column=1, columnspan=2, sticky="w", padx=10)
        self.fmt_stl = tk.BooleanVar(value=True)
        self.fmt_3mf = tk.BooleanVar(value=False)
        self.fmt_glb = tk.BooleanVar(value=False)
        tk.Checkbutton(ff, text=".STL", variable=self.fmt_stl).pack(side=tk.LEFT, padx=(0,10))
        tk.Checkbutton(ff, text=".3MF", variable=self.fmt_3mf).pack(side=tk.LEFT, padx=(0,10))
        tk.Checkbutton(ff, text=".GLB", variable=self.fmt_glb).pack(side=tk.LEFT, padx=(0,10))
        tk.Label(ff, text="(STL=universal, 3MF=PrusaSlicer, GLB=web)",
                 fg="gray", font=("Helvetica", 8)).pack(side=tk.LEFT)

        ttk.Separator(self.root, orient="horizontal").grid(
            row=6, column=0, columnspan=3, sticky="ew", padx=10, pady=6)
        self.flip_var = tk.BooleanVar(value=True)
        self.invert_var = tk.BooleanVar(value=False)
        tk.Checkbutton(self.root, text="Flip vertically (recommended - matches image orientation)",
                       variable=self.flip_var).grid(row=7, column=0, columnspan=3, sticky="w", padx=10, pady=2)
        tk.Checkbutton(self.root, text="Invert relief (dark areas raised - for some TEM images)",
                       variable=self.invert_var).grid(row=8, column=0, columnspan=3, sticky="w", padx=10, pady=2)

        ttk.Separator(self.root, orient="horizontal").grid(
            row=9, column=0, columnspan=3, sticky="ew", padx=10, pady=6)
        tk.Label(self.root, text="Parameters", font=("Helvetica", 11, "bold")).grid(
            row=10, column=0, columnspan=3, pady=(0,2))
        tk.Label(self.root, text="Smart defaults are set from your image. Only print width needed.",
                 font=("Helvetica", 9), fg="gray").grid(row=11, column=0, columnspan=3, pady=(0,6))

        sf = tk.LabelFrame(self.root, text="Smart defaults", font=("Helvetica", 9), padx=8, pady=6)
        sf.grid(row=12, column=0, columnspan=3, padx=14, pady=(0,4), sticky="ew")
        tk.Label(sf, text="Print width (mm):", anchor="w").grid(row=0, column=0, sticky="w", pady=3)
        self.size_x_var = tk.StringVar(value="100")
        sx = tk.Entry(sf, textvariable=self.size_x_var, width=8)
        sx.grid(row=0, column=1, sticky="w", padx=6)
        sx.bind("<FocusOut>", lambda e: self._update_smart_preview())
        self.smart_preview_var = tk.StringVar(value="")
        tk.Label(sf, textvariable=self.smart_preview_var,
                 fg="#2a7ae2", font=("Helvetica", 9)).grid(row=0, column=2, sticky="w", padx=6)
        self.smart_summary_var = tk.StringVar(value="Add images to see smart defaults.")
        tk.Label(sf, textvariable=self.smart_summary_var,
                 fg="gray", font=("Helvetica", 9), justify="left").grid(
            row=1, column=0, columnspan=3, sticky="w", pady=(4,0))

        ah = tk.Frame(self.root)
        ah.grid(row=13, column=0, columnspan=3, padx=14, pady=(6,0), sticky="w")
        tk.Checkbutton(ah, text="Full customization",
                       variable=self.advanced_open, font=("Helvetica", 10, "bold"),
                       command=self._toggle_advanced).pack(side=tk.LEFT)
        tk.Label(ah, text="Warning: override smart defaults - uncharted territory",
                 fg="#b05000", font=("Helvetica", 8)).pack(side=tk.LEFT, padx=6)

        self.adv_panel = tk.LabelFrame(self.root, text="Advanced parameters",
                                       font=("Helvetica", 9), padx=8, pady=6)
        adv_params = [
            ("Print height (mm)",   "size_y",        "auto", "auto = from aspect ratio"),
            ("Relief height (mm)",  "relief_height", "3.0",  "Max tactile height. Default: 3.0 mm"),
            ("Base thickness (mm)", "base_thickness","1.0",  "Solid base. Default: 1.0 mm"),
            ("Blur (sigma)",        "blur",          "1.2",  "Smoothing. Default: 1.2"),
            ("Resolution (px)",     "resolution",    "auto", "auto or 256 / 384 / 512"),
        ]
        self.adv_vars = {}
        for i, (label, key, default, hint) in enumerate(adv_params):
            tk.Label(self.adv_panel, text=label+":").grid(row=i, column=0, sticky="w", pady=2)
            var = tk.StringVar(value=default)
            self.adv_vars[key] = var
            tk.Entry(self.adv_panel, textvariable=var, width=10).grid(
                row=i, column=1, sticky="w", padx=6, pady=2)
            tk.Label(self.adv_panel, text=hint, fg="gray", font=("Helvetica", 8)).grid(
                row=i, column=2, sticky="w", padx=(0,8), pady=2)

        ttk.Separator(self.root, orient="horizontal").grid(
            row=15, column=0, columnspan=3, sticky="ew", padx=10, pady=6)
        self.progress = ttk.Progressbar(self.root, orient=tk.HORIZONTAL,
                                        length=420, mode="determinate")
        self.progress.grid(row=16, column=0, columnspan=3, padx=10, pady=4)
        self.status_var = tk.StringVar(value="Ready.")
        tk.Label(self.root, textvariable=self.status_var, fg="gray",
                 font=("Helvetica", 9)).grid(row=17, column=0, columnspan=3, pady=(0,4))
        self.run_btn = tk.Button(self.root, text="Generate files",
                                 font=("Helvetica", 11, "bold"),
                                 bg="#2a7ae2", fg="white",
                                 activebackground="#1a5fbf", activeforeground="white",
                                 command=self._run, width=20, height=2)
        self.run_btn.grid(row=18, column=0, columnspan=3, pady=(4,14))

    def _toggle_advanced(self):
        if self.advanced_open.get():
            self.adv_panel.grid(row=14, column=0, columnspan=3, padx=14, pady=(4,0), sticky="ew")
        else:
            self.adv_panel.grid_remove()

    def _add_images(self):
        files = filedialog.askopenfilenames(
            title="Select images",
            filetypes=[("Image files", "*.png *.jpg *.jpeg *.tif *.tiff *.bmp"),
                       ("All files", "*.*")])
        for f in files:
            if f not in self.image_paths:
                self.image_paths.append(f)
                self.listbox.insert(tk.END, Path(f).name)
        self._update_smart_preview()

    def _remove_selected(self):
        for i in reversed(list(self.listbox.curselection())):
            self.listbox.delete(i)
            self.image_paths.pop(i)
        self._update_smart_preview()

    def _browse_output(self):
        folder = filedialog.askdirectory()
        if folder:
            self.output_var.set(folder)

    def _update_smart_preview(self):
        if not self.image_paths:
            self.smart_summary_var.set("Add images to see smart defaults.")
            self.smart_preview_var.set("")
            return
        img_path = self.image_paths[0]
        try:
            size_x = float(self.size_x_var.get())
        except Exception:
            size_x = 100.0
        size_y = smart_size_y(img_path, size_x)
        res = smart_resolution(img_path)
        try:
            img = Image.open(img_path)
            w, h = img.size
        except Exception:
            w = h = "?"
        self.smart_preview_var.set(f"-> height = {size_y} mm")
        self.smart_summary_var.set(
            f"Image: {w}x{h} px | Print: {size_x}x{size_y} mm | "
            f"Res: {res} px | Relief: {BEST_PRACTICE['relief_height']} mm | "
            f"Blur: {BEST_PRACTICE['blur']}")

    def _get_params(self, img_path):
        try:
            size_x = float(self.size_x_var.get())
        except Exception:
            size_x = 100.0
        if self.advanced_open.get():
            try: size_y = float(self.adv_vars["size_y"].get())
            except Exception: size_y = smart_size_y(img_path, size_x)
            try: resolution = int(self.adv_vars["resolution"].get())
            except Exception: resolution = smart_resolution(img_path)
            try: relief = float(self.adv_vars["relief_height"].get())
            except Exception: relief = BEST_PRACTICE["relief_height"]
            try: base = float(self.adv_vars["base_thickness"].get())
            except Exception: base = BEST_PRACTICE["base_thickness"]
            try: blur = float(self.adv_vars["blur"].get())
            except Exception: blur = BEST_PRACTICE["blur"]
        else:
            size_y = smart_size_y(img_path, size_x)
            resolution = smart_resolution(img_path)
            relief = BEST_PRACTICE["relief_height"]
            base = BEST_PRACTICE["base_thickness"]
            blur = BEST_PRACTICE["blur"]
        return {"size_x": size_x, "size_y": size_y, "relief_height": relief,
                "base_thickness": base, "blur": blur, "resolution": resolution,
                "invert": self.invert_var.get(), "flip": self.flip_var.get()}

    def _get_formats(self):
        f = {"stl": self.fmt_stl.get(), "3mf": self.fmt_3mf.get(), "glb": self.fmt_glb.get()}
        if not any(f.values()):
            messagebox.showwarning("No format", "Select at least one output format.")
            return None
        return f

    def _run(self):
        if not self.image_paths:
            messagebox.showwarning("No images", "Add at least one image.")
            return
        formats = self._get_formats()
        if formats is None or self.running:
            return
        self.running = True
        self.run_btn.config(state=tk.DISABLED)
        threading.Thread(target=self._process_all, args=(formats,), daemon=True).start()

    def _process_all(self, formats):
        output_dir = Path(self.output_var.get())
        output_dir.mkdir(parents=True, exist_ok=True)
        total = len(self.image_paths)
        errors = []
        for i, img_path in enumerate(self.image_paths):
            name = Path(img_path).stem
            self._set_status(f"Processing {i+1}/{total}: {name}...")
            self.progress["value"] = (i / total) * 100
            self.root.update_idletasks()
            try:
                params = self._get_params(img_path)
                hm = image_to_heightmap(
                    img_path,
                    output_size=(params["resolution"], params["resolution"]),
                    invert=params["invert"], blur_sigma=params["blur"],
                    preserve_aspect=True, flip=params["flip"])
                stl_path = output_dir / f"{name}_lithograph.stl"
                heightmap_to_stl(hm, stl_path,
                    base_thickness_mm=params["base_thickness"],
                    max_relief_mm=params["relief_height"],
                    physical_size_mm=(params["size_x"], params["size_y"]))
                convert_stl(stl_path, formats)
            except Exception as e:
                errors.append(f"{name}: {e}")
        self.progress["value"] = 100
        fmt_list = ", ".join(f".{k.upper()}" for k, v in formats.items() if v)
        if errors:
            self._set_status(f"Done with {len(errors)} error(s).")
            messagebox.showerror("Errors", "Some files failed:\n\n" + "\n".join(errors))
        else:
            self._set_status(f"Done! {total} file(s) as {fmt_list}")
            messagebox.showinfo("Done!",
                f"Generated {total} file(s) as {fmt_list}.\n\nSaved to:\n{output_dir}")
        self.running = False
        self.run_btn.config(state=tk.NORMAL)

    def _set_status(self, msg):
        self.status_var.set(msg)
        self.root.update_idletasks()


# Launch
root = tk.Tk()
app = TAMPBatchGUI(root)
root.mainloop()
